[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb)

# 🟡 Medium: Implement LayerNorm

Implement **Layer Normalization** from scratch.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu$ and $\sigma^2$ are computed over the **last dimension**.

### Signature
```python
def my_layer_norm(
    x: torch.Tensor,      # input
    gamma: torch.Tensor,   # scale (same size as last dim)
    beta: torch.Tensor,    # shift (same size as last dim)
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use `F.layer_norm` or `torch.nn.LayerNorm`
- Normalize over the last dimension only
- Must support autograd

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch

In [7]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_layer_norm(x, gamma, beta, eps=1e-5):
    mean = torch.mean(x, dim=-1, keepdim=True)
    #std  = torch.std(x, dim=-1, keepdim=True) \
    # correction: very important!
    std  = torch.std(x, dim=-1, keepdim=True, correction=0)
    # or directly use torch.var

    print(x.shape, mean.shape, std.shape, gamma.shape)

    return (x - mean) / torch.sqrt(torch.pow(std, 2) + eps) * gamma + beta


In [8]:
# 🧪 Debug
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("Your output mean:", out.mean(dim=-1))   # should be ~0
print("Your output std: ", out.std(dim=-1))     # should be ~1
print(out, ref)
print("Match ref?      ", torch.allclose(out, ref, atol=1e-4))

torch.Size([2, 8]) torch.Size([2, 1]) torch.Size([2, 1]) torch.Size([8])
Your output mean: tensor([-7.4506e-09, -3.7253e-08])
Your output std:  tensor([1.0690, 1.0690])
tensor([[-1.1721, -0.4049,  0.6539,  1.4444, -0.1297, -1.4216,  1.3457, -0.3158],
        [ 1.0795, -0.7523, -1.2323,  1.4896, -0.2270,  1.1538, -0.6999, -0.8114]]) tensor([[-1.1721, -0.4049,  0.6539,  1.4444, -0.1297, -1.4216,  1.3457, -0.3158],
        [ 1.0795, -0.7523, -1.2323,  1.4896, -0.2270,  1.1538, -0.6999, -0.8114]])
Match ref?       True


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("layernorm")